In [3]:
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

In [4]:
X, y = make_classification(n_samples=10000, n_features=10, n_informative=3) 

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [8]:
dt = DecisionTreeClassifier(random_state = 42)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)

print("Decision Tree accuracy :", accuracy_score(y_test, y_pred))

Decision Tree accuracy : 0.8685


### Bagging :

In [10]:
bag = BaggingClassifier (
    estimator = DecisionTreeClassifier(),
    n_estimators = 500,
    max_samples = 0.25,
    bootstrap = True,
    random_state = 42
               
)

In [11]:
bag.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.25,
                  n_estimators=500, random_state=42)

In [12]:
y_pred = bag.predict(X_test)

In [13]:
accuracy_score(y_test, y_pred)

0.9225

In [14]:
bag.estimators_samples_[0].shape

(2000,)

In [15]:
bag.estimators_features_[0].shape

(10,)

### Bagging with SVM :

In [17]:
bag = BaggingClassifier(
     estimator = SVC(),
    n_estimators = 500,
    max_samples = 0.25,
    bootstrap = True, 
    random_state = 42  )
               

In [18]:
bag.fit(X_train, y_train)
y_pred = bag.predict(X_test)
print("Bagging using SVM :", accuracy_score(y_test, y_pred))

Bagging using SVM : 0.899


### Pasting :

In [20]:
bag = BaggingClassifier(
    estimator = DecisionTreeClassifier(),
    n_estimators = 500,
    max_samples = 0.25,
    bootstrap = False,
    random_state = 42,
    verbose = 1,   # Controls the amount of messages printed while training. 0 → silent, 1 → prints progress (like “fitting estimator 1/500”),
    n_jobs = -1  )  # Defines how many CPU cores are used in parallel. -1 means use all available cores → much faster training for 500 trees.
               

In [21]:
bag.fit(X_train, y_train)
y_pred = bag.predict(X_test)
print("Pasting Classifier", accuracy_score(y_test, y_pred))

[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   2 out of  12 | elapsed:    8.2s remaining:   41.3s
[Parallel(n_jobs=12)]: Done  12 out of  12 | elapsed:    8.4s finished
[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   2 out of  12 | elapsed:    0.0s remaining:    0.4s
[Parallel(n_jobs=12)]: Done  12 out of  12 | elapsed:    0.2s finished


Pasting Classifier 0.9235


### Random Subspaces : (column sampling)

In [23]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=1.0,
    bootstrap=False,
    max_features=0.5,
    bootstrap_features=True,
    random_state=42
) 

In [24]:
bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print("Random Subspaces classifier",accuracy_score(y_test,y_pred))

Random Subspaces classifier 0.9115


In [25]:
bag.estimators_samples_[0].shape    

(8000,)

In [26]:
bag.estimators_features_[0].shape

(5,)

### Random Patches : (Rows and Columns sampling)

In [28]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    max_features=0.5,
    bootstrap_features=True,
    random_state=42
)

In [29]:
bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print("Random Patches classifier",accuracy_score(y_test,y_pred))
     

Random Patches classifier 0.916


#### OOB Score : (Out-of-Bag)

In [32]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    oob_score=True,
    random_state=42
)

In [33]:
bag.fit(X_train,y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.25,
                  n_estimators=500, oob_score=True, random_state=42)

In [34]:
bag.oob_score_

0.920625

In [35]:
y_pred = bag.predict(X_test)
print("Accuracy",accuracy_score(y_test,y_pred))

Accuracy 0.9225


### Bagging Tips
- Bagging generally gives better results than Pasting
- Good results come around the 25% to 50% row sampling mark
- Random patches and subspaces should be used while dealing with high dimensional data
- To find the correct hyperparameter values we can do GridSearchCV/RandomSearchCV

### Applying GridSearchCV :

In [38]:
from sklearn.model_selection import GridSearchCV

In [39]:
parameters = {
    'n_estimators': [50,100,500,1000], 
    'max_samples': [0.1,0.4,0.7,1.0],
    'bootstrap' : [True,False],
    'max_features' : [0.1,0.4,0.7,1.0]
    }

In [40]:
search = GridSearchCV(BaggingClassifier(), parameters, cv=5)


In [41]:
search.fit(X_train,y_train)
     

GridSearchCV(cv=5, estimator=BaggingClassifier(),
             param_grid={'bootstrap': [True, False],
                         'max_features': [0.1, 0.4, 0.7, 1.0],
                         'max_samples': [0.1, 0.4, 0.7, 1.0],
                         'n_estimators': [50, 100, 500, 1000]})

In [ ]:
search.best_params_
search.best_score_
     

In [ ]:
search.best_params_